In [1]:
import os
import numpy as np
import supervision as sv
import torch
from PytorchWildlife.models import detection as pw_detection
from PytorchWildlife.models import classification as pw_classification
from PytorchWildlife import utils as pw_utils

from speciesnet import DEFAULT_MODEL
from speciesnet import draw_bboxes
from speciesnet import load_rgb_image
from speciesnet import SpeciesNet
from speciesnet import SUPPORTED_MODELS

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.cuda.set_device(0)
SOURCE_VIDEO_PATH = "../../data/test/IMG_2473.MP4"
TARGET_VIDEO_PATH = "../../data/test/IMG_2473_processed.MP4"
print(f"Using deice {DEVICE}")

Using deice cuda


In [3]:
print("Default SpeciesNet model:", DEFAULT_MODEL)
print("Supported SpeciesNet models:", SUPPORTED_MODELS)
model = SpeciesNet(DEFAULT_MODEL)

Default SpeciesNet model: kaggle:google/speciesnet/pyTorch/v4.0.2a/1
Supported SpeciesNet models: ['kaggle:google/speciesnet/pyTorch/v4.0.2a/1', 'kaggle:google/speciesnet/pyTorch/v4.0.2b/1']


In [5]:
instances = {"instances": [
    {
        "filepath": "../../data/test/IMG_2473.MP4"
    }
]
}
speciesnet_pred = model.predict(instances_dict=instances)

ERROR:absl:`UnidentifiedImageError` while loading `../../data/test/IMG_2473.MP4` ==> cannot identify image file '../../data/test/IMG_2473.MP4'
ERROR:absl:`UnidentifiedImageError` while loading `../../data/test/IMG_2473.MP4` ==> cannot identify image file '../../data/test/IMG_2473.MP4'


In [ ]:
# Valid versions are MDV6-yolov9-c, MDV6-yolov9-e, MDV6-yolov10-c, MDV6-yolov10-e or MDV6-rtdetr-c
detection_model = pw_detection.MegaDetectorV6(device=DEVICE, pretrained=True, version="MDV6-yolov10-e")
classification_model = pw_classification.AI4GOpossum(device=DEVICE, pretrained=True)

Ultralytics 8.4.14 🚀 Python-3.11.14 torch-2.10.0+cu128 CUDA:0 (NVIDIA A40, 45488MiB)
YOLOv10x summary (fused): 191 layers, 29,399,417 parameters, 0 gradients, 160.0 GFLOPs


In [12]:
box_annotator = sv.BoxAnnotator(thickness=4)
lab_annotator = sv.LabelAnnotator(text_color=sv.Color.BLACK, text_thickness=4, text_scale=2)

def callback(frame: np.ndarray, index: int) -> np.ndarray:
    results_det = detection_model.single_image_detection(frame, img_path=index)
    labels = []
    for xyxy in results_det["detections"].xyxy:
        cropped_image = sv.crop_image(image=frame, xyxy=xyxy)
        results_clf = classification_model.single_image_classification(cropped_image)
        labels.append("{} {:.2f}".format(results_clf["prediction"], results_clf["confidence"]))
    annotated_frame = lab_annotator.annotate(
        scene=box_annotator.annotate(
            scene=frame,
            detections=results_det["detections"],
        ),
        detections=results_det["detections"],
        labels=labels,
    )
    return annotated_frame 

pw_utils.process_video(source_path=SOURCE_VIDEO_PATH, target_path=TARGET_VIDEO_PATH, callback=callback, target_fps=5)

  0%|          | 0/50 [00:00<?, ?it/s]


0: 1280x1280 1 animal, 44.3ms
Speed: 257.3ms preprocess, 44.3ms inference, 311.5ms postprocess per image at shape (1, 3, 1280, 1280)


  2%|▏         | 1/50 [00:27<22:09, 27.14s/it]


0: 1280x1280 1 animal, 44.3ms
Speed: 5.7ms preprocess, 44.3ms inference, 0.4ms postprocess per image at shape (1, 3, 1280, 1280)


  4%|▍         | 2/50 [00:27<09:00, 11.25s/it]

...

 96%|█████████▌| 48/50 [00:30<00:00, 12.20it/s]


0: 1280x1280 1 animal, 44.1ms
Speed: 4.4ms preprocess, 44.1ms inference, 0.4ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 2 animals, 44.2ms
Speed: 4.3ms preprocess, 44.2ms inference, 0.4ms postprocess per image at shape (1, 3, 1280, 1280)


100%|██████████| 50/50 [00:31<00:00, 12.32it/s]


0: 1280x1280 2 animals, 44.2ms
Speed: 4.3ms preprocess, 44.2ms inference, 0.4ms postprocess per image at shape (1, 3, 1280, 1280)


51it [00:31,  1.63it/s]                        
